# MERIT — Stage 5: the real-data test

**This is the make-or-break experiment.** Read `PREREGISTRATION.md` first.

Order matters:

1. Sections 0–2 are free.
2. Section 3 is a small live smoke test (~15–20k tokens) on the *development* conversation.
3. **Commit `PREREGISTRATION.md` and the code, fill in the commit hash, and push — before section 5.**
4. Sections 5–6 are the confirmatory run (~1.4–2.1M tokens, resumable across days).

Do not change any setting in `SETTINGS` after registering.

In [ ]:
import sys, pathlib, subprocess
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.width", 200)
from merit import locomo as LO, stage5 as S5

SETTINGS = dict(            # must match PREREGISTRATION.md section 5
    split_seed=20261006, seed=20261006,
    model="openai/gpt-oss-20b", reasoning_effort="low", max_tokens=512,
    top_k=5, epochs=4, mask_p=0.30, keep_frac=0.30,
)
DEV_CONV = 7
PRIMARY  = [3, 4, 9, 8]     # auto-capture, H1 and H2
BOUNDARY = [3, 2]           # clean store, B1
ARMS     = ("utility", "recall", "random")   # as registered

MONITOR_EVERY = 10    # refresh the dashboard every N turns (3 is fine too, just busier)
BLIND = True          # hide held-out arm F1 until every primary conversation is done
DB = "data/stage5.db"
data = LO.load_locomo()

from dotenv import load_dotenv               # keys load here, so a fresh kernel can
load_dotenv(ROOT / ".env")                   # jump straight to section 5 on day 2

## 0. Registration check

In [ ]:
print((ROOT / "PREREGISTRATION.md").read_text()[:600], "...")
try:
    h = subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
    print("\ncurrent commit:", h)
    print("uncommitted changes:", "YES — commit before section 5" if dirty else "none")
except Exception as e:
    print("git not available:", e)

## 1. Selections follow the stated rule

Recomputed here so nobody — including you — can quietly change them.

In [ ]:
rows = []
for i in range(len(data)):
    for track, inc in (("clean", False), ("autocapture", True)):
        cv = LO.build_conversation(data, i, include_turns=inc)
        learn, test = LO.split_tasks(cv.tasks, seed=SETTINGS["split_seed"])
        rows.append(dict(conv=i, track=track, store=len(cv.store), eligible=len(cv.tasks),
                         test=len(test), **{k: v for k, v in LO.transfer_ceiling(cv, learn, test).items()
                                            if k != "test_questions"}))
sel = pd.DataFrame(rows)
rule_primary  = (sel[(sel.track=="autocapture") & (sel.conv!=DEV_CONV)]
                 .sort_values(["eligible","conv"], ascending=[False,True]).conv.head(4).tolist())
rule_boundary = (sel[(sel.track=="clean") & (sel.conv!=DEV_CONV)]
                 .sort_values(["eligible","conv"], ascending=[False,True]).conv.head(2).tolist())
assert rule_primary == PRIMARY,   (rule_primary, PRIMARY)
assert rule_boundary == BOUNDARY, (rule_boundary, BOUNDARY)
print("selections match the pre-registered rule")
sel[sel.conv.isin(set(PRIMARY + BOUNDARY))]

`all_evidence_seen` is the transfer ceiling: the share of held-out questions
whose evidence any learning question used. MERIT has nothing to learn from on
the rest, by construction.

## 2. Development replication — mock, conversation 7, zero tokens

Reproduces what motivated the design. Clean store: utility loses. Auto-capture
store: utility leads, not significantly. This is development evidence only.

In [ ]:
dev = {}
for track in ("clean", "autocapture"):
    r = S5.run_conversation(data, DEV_CONV, track=track, backend="mock", settings=SETTINGS,
                            arms=("utility","recall","random","full"),
                            db_path="data/stage5_dev_mock.db", verbose=False,
                            monitor_every=0)
    dev[track] = r
    print(f"\n=== {track}: {r['store_entries']} entries ===  evidence kept: {r['evidence_kept']}")
    print(S5.arm_table(r["turns"])[["mean_f1","store_size"]].to_string())
    print(S5.paired_bootstrap(r["turns"], "utility", "recall"))

## 3. Live smoke test — development conversation, ~15–20k tokens

Purpose: confirm the real model answers sensibly with this prompt, measure real
tokens per turn, and check the empty-completion rate with `reasoning_effort="low"`.
Uses only conversation 7, so it cannot contaminate the confirmatory run.

In [ ]:
conv = LO.build_conversation(data, DEV_CONV, include_turns=True)
learn, _ = LO.split_tasks(conv.tasks, seed=SETTINGS["split_seed"])
smoke = S5.learn_phase(conv, learn[:40], backend="groq", epochs=1, mask_p=0.0,
                       seed=SETTINGS["seed"], run_tag="_smoke", db_path="data/stage5_smoke.db",
                       model=SETTINGS["model"], reasoning_effort=SETTINGS["reasoning_effort"],
                       max_tokens=SETTINGS["max_tokens"], max_total_tokens=40_000, verbose=True,
                       monitor_every=5)
from merit import Ledger
from merit.config import Config
lg = Ledger(Config(db_path="data/stage5_smoke.db").resolved_db_path())
st = lg.turns_dataframe(f"s5_learn_c{DEV_CONV}_smoke"); lg.close()
print("tokens/turn:", round(st.turn_tokens.mean()), "| prompt:", round(st.prompt_tokens.mean()),
      "| completion:", round(st.completion_tokens.mean()))
print("empty completions:", int(st.outcome_score.isna().sum()), "/", len(st))
print("mean F1 (full store, no masking):", round(st.outcome_score.mean(), 3))

In [ ]:
gold = {t.task_id: t.gold for t in learn}
st.assign(gold=st.task_id.map(gold))[["task_id","gold","response_text","outcome_score","turn_tokens"]].head(15)

**Read those fifteen by hand.** You're checking that answers are short phrases
(F1 punishes sentences), that the model is using the notes, and that F1 isn't
unfairly zeroing correct answers (e.g. "May 7, 2023" vs gold "7 May 2023" scores
partial credit). If answers are long sentences, the prompt needs fixing — **now**,
before registration, not after.

If the mean F1 here is near zero, stop: the pipeline is broken, not MERIT.

## 4. Budget, calibrated from the smoke test

In [ ]:
ORGS = 1        # how many SEPARATE Groq accounts your keys come from (limits are per account)
TPD, RPD = 200_000, 1_000                # free tier, per account, gpt-oss-20b

per_turn = st.turn_tokens.mean()
plan = [("autocapture", c) for c in PRIMARY] + [("clean", c) for c in BOUNDARY]
total = 0
for track, c in plan:
    row = sel[(sel.conv == c) & (sel.track == track)].iloc[0]
    learn_n = row.eligible // 2
    turns = SETTINGS["epochs"] * learn_n + len(ARMS) * row.test
    total += turns
    print(f"{track:12s} conv {c}: {turns:4d} turns")
tokens = int(total * per_turn)
days = max(tokens / (TPD * ORGS), total / (RPD * ORGS))
print(f"\ntotal {total} turns ≈ {tokens:,} tokens at {per_turn:.0f} tokens/turn")
print(f"≈ {days:.1f} days at {ORGS} account(s) on the free tier")

## 5. Confirmatory run — STOP and register first

Checklist before running this section:

- [ ] Section 3 answers looked right
- [ ] `PREREGISTRATION.md` has the commit hash and date filled in
- [ ] `git add -A && git commit && git push` done
- [ ] Section 0 reports no uncommitted changes

Each conversation is resumable. If a key runs dry or you stop the kernel,
re-run the same cell and finished phases are reused.

**When the daily limit hits**, the cell stops with a `Paused` message. Nothing
is lost. Re-run the same cell after the reset (Groq resets daily) and it
continues from the exact turn where it stopped.

**Day 2 onward:** restart the kernel, run only the setup cell at the top, then
jump straight here. Don't re-run section 3 — it spends tokens.
If the kernel was restarted after everything finished, re-running these cells
reuses all saved results instantly, at zero cost.

### Where am I? — run this any time

Shows every phase of the whole experiment: done / total, tokens, empty answers.
Learning-phase F1 is shown; held-out arm F1 stays `hidden` while `BLIND = True`.
Costs nothing — it only reads the database.

In [ ]:
PLAN = [("autocapture", c) for c in PRIMARY] + [("clean", c) for c in BOUNDARY]
S5.status(DB, data, SETTINGS, PLAN, arms=ARMS, blind=BLIND)

In [ ]:
RESULTS = {}
for c in PRIMARY:
    RESULTS[("autocapture", c)] = S5.run_conversation(
        data, c, track="autocapture", backend="groq", settings=SETTINGS, arms=ARMS, db_path=DB,
        monitor_every=MONITOR_EVERY, blind=BLIND)
    RESULTS[("autocapture", c)]["turns"].to_csv(ROOT / f"data/stage5_autocapture_c{c}.csv", index=False)
    print(f"conv {c} done:", RESULTS[("autocapture", c)]["evidence_kept"])

In [ ]:
for c in BOUNDARY:
    RESULTS[("clean", c)] = S5.run_conversation(
        data, c, track="clean", backend="groq", settings=SETTINGS, arms=ARMS, db_path=DB,
        monitor_every=MONITOR_EVERY, blind=BLIND)
    RESULTS[("clean", c)]["turns"].to_csv(ROOT / f"data/stage5_clean_c{c}.csv", index=False)
    print(f"conv {c} done:", RESULTS[("clean", c)]["evidence_kept"])

## 6. Analysis — exactly as registered

In [ ]:
prim = pd.concat([RESULTS[("autocapture", c)]["turns"] for c in PRIMARY], ignore_index=True)
bnd  = pd.concat([RESULTS[("clean", c)]["turns"] for c in BOUNDARY], ignore_index=True)

h1 = S5.paired_bootstrap(prim, "utility", "recall", seed=SETTINGS["seed"])
h2 = S5.paired_bootstrap(prim, "utility", "random", seed=SETTINGS["seed"])
b1 = S5.paired_bootstrap(bnd,  "utility", "recall", seed=SETTINGS["seed"])
pd.DataFrame([dict(id="H1 (primary)", **h1), dict(id="H2", **h2), dict(id="B1 boundary", **b1)])

In [ ]:
def verdict(r):
    if r["ci_lo"] > 0:  return "SUPPORTED"
    if r["ci_hi"] < 0:  return "REFUTED"
    return "INCONCLUSIVE"
print("H1 (utility vs recall, auto-capture):", verdict(h1),
      f"  {h1['mean_diff']:+.4f}  [{h1['ci_lo']:+.4f}, {h1['ci_hi']:+.4f}]")
print("H2 (utility vs random, auto-capture):", verdict(h2))
print("B1 (utility vs recall, clean store)  :", f"{b1['mean_diff']:+.4f}  [{b1['ci_lo']:+.4f}, {b1['ci_hi']:+.4f}]",
      "(no directional prediction)")

### Per-conversation detail and what each rule kept

Reported for transparency. The verdict above is the pooled H1 result only.

In [ ]:
rows = []
for (track, c), r in RESULTS.items():
    tab = S5.arm_table(r["turns"])
    for arm in tab.index:
        rows.append(dict(track=track, conv=c, arm=arm, mean_f1=tab.loc[arm, "mean_f1"],
                         evidence_kept=r["evidence_kept"].get(arm, ""),
                         transfer_all=r["transfer"]["all_evidence_seen"],
                         withheld5=r["ledger_quality"]["entries_with_5plus_withheld"]))
detail = pd.DataFrame(rows)
detail.to_csv(ROOT / "data/stage5_detail.csv", index=False)
detail.pivot_table(index=["track","conv"], columns="arm", values="mean_f1")